# Mini Project : Text 2 SQL
__Team: P2D2 (Prompt 2 Data-Demand)__
<br>_Members:_
* _Jeanne Malécot_
* _Arthur Nuvoloni_
* _Adam Sebti_
* _Benjamin Ternot_

> Dataset : https://huggingface.co/datasets/xlangai/spider

In [1]:
!pip install pandas numpy torch tf-keras
!python3 -m pip install scikit-learn
!pip install peft datasets evaluate transformers[sentencepiece] bitsandbytes

import os
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import evaluate
import pandas as pd
import numpy as np
import torch
from tensorflow.python.client import device_lib


2025-01-06 13:41:32.309637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736167292.329544 1799883 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736167292.335696 1799883 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-06 13:41:32.358326: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')

print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 17829326108719178676
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 15502344192
locality {
  bus_id: 1
  links {
    link {
      device_id: 1
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 5503962068381501663
physical_device_desc: "device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:3b:00.0, compute capability: 7.0"
xla_global_id: 416903419
, name: "/device:GPU:1"
device_type: "GPU"
memory_limit: 32646561792
locality {
  bus_id: 1
  links {
    link {
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 140362577740227533
physical_device_desc: "device: 1, name: Tesla V100-PCIE-32GB, pci bus id: 0000:af:00.0, compute capability: 7.0"
xla_global_id: 2144165316
]


I0000 00:00:1736167296.665361 1799883 gpu_device.cc:2022] Created device /device:GPU:0 with 14784 MB memory:  -> device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1736167296.666071 1799883 gpu_device.cc:2022] Created device /device:GPU:1 with 31134 MB memory:  -> device: 1, name: Tesla V100-PCIE-32GB, pci bus id: 0000:af:00.0, compute capability: 7.0


In [8]:
# Huggingface token
with open('huggingface_token.txt', 'r') as f:
    tokens=[line for line in f]
huggingface_token = tokens[0]

# Load the dataset
dataset = load_dataset("xlangai/spider", split="train")
val_dataset = load_dataset("xlangai/spider", split="validation")

# Split train into train and test
train_dataset, test_dataset = dataset.train_test_split(test_size=0.2, shuffle=True, seed=42).values()

# Load the pre-trained model and tokenizer
model_name = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=huggingface_token, device_map="cuda")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_8bit=True, device_map="cuda", token=huggingface_token)


# Define LoRA Config
lora_config = LoraConfig(
    r=8, # Rank of the LoRA update matrices
    lora_alpha=32, # Scaling factor for the LoRA updates
    lora_dropout=0.05, # Dropout probability for the LoRA layers
    bias="none",  # Whether to apply a bias to the LoRA layers
    task_type="SEQ_2_SEQ_LM"  # Type of task for which the model is being fine-tuned
)

# Apply LoRA to the model using peft
model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [9]:
# Preprocessing function
def preprocess_function(examples):
    # Combine question and schema into a single input
    inputs = [f"Question: {q}\nSchema: {s}\nSQL:" for q, s in zip(examples["question"], examples["db_id"])]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length') # Add padding='max_length'

    with tokenizer.as_target_tokenizer():
      labels = tokenizer(examples["query"], max_length=512, truncation=True, padding='max_length') # Add padding='max_length'

    # Create attention mask: 1 for real tokens, 0 for padding
    model_inputs["attention_mask"] = [[1] * len(input_ids) + [0] * (512 - len(input_ids)) for input_ids in model_inputs['input_ids']]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_datasets = train_dataset.map(preprocess_function, batched=True)
tokenized_val_datasets = val_dataset.map(preprocess_function, batched=True)

# Define the metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Calculate the metrics
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"accuracy": result["accuracy"]}

# Define training arguments
training_args = TrainingArguments(
    output_dir="llama3-sql-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_accumulation_steps=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=10,
    push_to_hub=False,
    fp16=True,
    dataloader_num_workers=4,  # Use multiple workers for faster data loading
    report_to="none",
    ddp_find_unused_parameters=False,  # Helps with Distributed Data Parallel (DDP)
)

# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_datasets,
    eval_dataset=tokenized_val_datasets,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

Map:   0%|          | 0/5600 [00:00<?, ? examples/s]

/home/infres/bternot-21/Text2SQL/venv-Text2SQL/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

/home/infres/bternot-21/Text2SQL/venv-Text2SQL/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_1799883/1308070575.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# Fine-tune the model
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # Add this line to enable more detailed error messages
trainer.train()

TypeError: device() received an invalid combination of arguments - got (NoneType), but expected one of:
 * (torch.device device)
      didn't match because some of the arguments have invalid types: (!NoneType!)
 * (str type, int index = -1)


In [ ]:
# Evaluation Function (Placeholders for actual evaluation logic)
def evaluate_model(model, test_data):
    # 1. Valid answer ratio
    valid_answers = 0
    # 2. Exact Match Accuracy (EMA)
    exact_matches = 0
    # 3. Execution Accuracy (EA)
    execution_accuracy = 0
    # 4. Syntax validity
    syntax_valid = 0

    # Placeholder - implement your model evaluation logic here.
    # This should predict SQL queries based on the test data,
    # and compare the predictions against the ground truth.

    num_tests = len(test_data)
    valid_answer_ratio = valid_answers / num_tests if num_tests > 0 else 0
    ema = exact_matches / num_tests if num_tests > 0 else 0
    ea = execution_accuracy / num_tests if num_tests > 0 else 0
    syntax_validity = syntax_valid / num_tests if num_tests > 0 else 0
    return valid_answer_ratio, ema, ea, syntax_validity